# GroundLens vs. the field — a reproducible hallucination-detection benchmark

This notebook benchmarks **GroundLens** against comparable hallucination / RAG-grounding
detectors on two public datasets, and reports the metrics that matter for a compliance
setting.

**What is actually comparable.** Most "eval" names in this space — LangSmith, Arize
Phoenix, Braintrust, Ragas, DeepEval — are *platforms* that run an **LLM-as-judge**; they
ship no hallucination model, so you cannot benchmark the platform, only the judge it runs.
We therefore represent all of them with a single **GPT-4o-as-judge** row. The tools that
*are* scoreable detectors (Vectara HHEM, MiniCheck, LettuceDetect, Patronus Lynx) are the
real head-to-head. Guardrail frameworks (NeMo Guardrails, Guardrails AI, Llama Guard) are a
different category (policy/safety enforcement), not grounding, and are out of scope.

**Run here (live):** GroundLens, Vectara **HHEM-2.1-Open** (the reference open detector),
and **GPT-4o-as-judge** (optional, needs an OpenAI key). HHEM/MiniCheck/LettuceDetect/Lynx
numbers from their own papers are cited in the accompanying analysis.

**Datasets:** **LLM-AggreFact** (claim-level, one-line load, has a public leaderboard) and
**RAGTruth** (span-level, RAG-native, MIT).

**Metrics:** **FPR @ 95% recall** (the headline: at the recall a reviewer demands, how many
false alarms?), plus **AUROC** and **balanced accuracy** for comparability.

> GroundLens's differentiators are not just accuracy: it is deterministic, emits a
> **signed, append-only evidence record**, localises to **spans**, and maps outcomes to
> **EU AI Act** articles. None of the detectors below produce an auditable evidence trail.


## 1 · Setup

Install GroundLens and the benchmark dependencies, then pull the `base` bundle (the multilingual encoder + the entailment model that power the lexical, semantic and NLI verifiers).

In [ ]:
# GroundLens + baselines. transformers is capped BELOW 5.0 on purpose: HHEM-2.1-Open
# ships custom model code on the Hub that breaks on transformers 5.x (the
# 'all_tied_weights_keys' error). The last 4.x line has Python-3.13 wheels, so
# nothing gets compiled from source. tokenizers is left unpinned so pip picks a
# wheel that matches.
%pip -q install "groundlens>=5.3" "transformers>=4.46,<5" \
    datasets scikit-learn matplotlib pandas tqdm sentencepiece

# IMPORTANT: if Colab had already imported transformers 5.x, RESTART THE RUNTIME
# now (Runtime > Restart session) and run every cell again from the top. The old
# version stays in memory until you restart, and HHEM will keep failing.


In [ ]:
# Pull the GroundLens base bundle once (encoder + entailment model). ~network download.
!groundlens bundle pull base
!groundlens bundle status

## 2 · Configuration

Keep the sample small for a free runtime; raise `SAMPLE_SIZE` on a GPU. The GPT-4o judge is optional and needs an OpenAI API key.

In [ ]:
import os, json, math, urllib.request, random, gc
random.seed(0)

SAMPLE_SIZE = 300          # per dataset
TARGET_RECALL = 0.95       # every operating-point metric is measured here
CONTEXT_CHARS = 4000       # cap context length; both detectors truncate well below this

# Optional GPT-4o-as-judge (stands in for LangSmith / Phoenix / Braintrust / Ragas / DeepEval).
RUN_LLM_JUDGE = False      # set True and provide a key to include the judge row
OPENAI_MODEL = "gpt-4o"
# In Colab: from google.colab import userdata; os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Cost column: GPT-4o price in USD per 1M tokens. Edit to your current rates.
# GroundLens and HHEM run locally, so their per-call cost is ~0.
PRICE_IN_PER_M = 2.50
PRICE_OUT_PER_M = 10.00


## 3 · Load the datasets

Both are cast to the same shape: `(context, claim, y)` where **`y = 1` means the claim is
*hallucinated / not supported*** by the context (the positive class we want to detect).

In [ ]:
from datasets import load_dataset
from itertools import islice

def load_llm_aggrefact(n):
    """LLM-AggreFact via streaming: only a bounded buffer is held in RAM, never the
    whole test split. Positive class = unsupported. ctype = sub-dataset name."""
    ds = load_dataset("lytang/LLM-AggreFact", split="test", streaming=True)
    ds = ds.shuffle(seed=0, buffer_size=8000)
    rows = []
    for r in islice(ds, n):
        rows.append({
            "context": r["doc"],
            "claim": r["claim"],
            "question": None,
            "y": 0 if int(r["label"]) == 1 else 1,
            "ctype": r.get("dataset", "LLM-AggreFact"),
        })
    del ds; gc.collect()
    return rows

llm_aggrefact = load_llm_aggrefact(SAMPLE_SIZE)
print("LLM-AggreFact:", len(llm_aggrefact), "examples;",
      sum(r["y"] for r in llm_aggrefact), "hallucinated")


In [ ]:
RAGTRUTH_BASE = "https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/"

def _read_jsonl_url(url):
    with urllib.request.urlopen(url) as f:
        return [json.loads(line) for line in f if line.strip()]

def _context_from_source(si):
    info = si.get("source_info", si)
    if isinstance(info, dict):
        for k in ("passages", "context", "document", "documents"):
            if k in info and info[k]:
                v = info[k]
                return "\n\n".join(v) if isinstance(v, list) else str(v)
        return json.dumps(info, ensure_ascii=False)
    return str(info)

def _question_from_source(si):
    info = si.get("source_info", si)
    return info.get("question") if isinstance(info, dict) else None

def load_ragtruth(n, task_type="QA"):
    """RAGTruth: join responses to their source; hallucinated = has labelled spans.
    ctype = the set of span label_types (Evident/Subtle Conflict, Baseless Info, ...)."""
    sources = {s["source_id"]: s for s in _read_jsonl_url(RAGTRUTH_BASE + "source_info.jsonl")}
    responses = _read_jsonl_url(RAGTRUTH_BASE + "response.jsonl")
    rows = []
    for r in responses:
        s = sources.get(r["source_id"])
        if s is None:
            continue
        if task_type and s.get("task_type") != task_type:
            continue
        labels = r.get("labels") or []
        types = sorted({(l.get("label_type") or "unknown") for l in labels}) if labels else []
        rows.append({
            "context": _context_from_source(s),
            "claim": r["response"],
            "question": _question_from_source(s),
            "y": 1 if labels else 0,
            "ctype": types,   # list; empty for non-hallucinated
        })
    random.shuffle(rows)
    out = rows[:n]
    del sources, responses, rows; gc.collect()
    return out

try:
    ragtruth = load_ragtruth(SAMPLE_SIZE, task_type="QA")
    print("RAGTruth (QA):", len(ragtruth), "examples;",
          sum(r["y"] for r in ragtruth), "hallucinated")
except Exception as e:
    print("RAGTruth load failed (", e, ") - continuing with LLM-AggreFact only.")
    ragtruth = []


## 4 · Detectors

Each detector maps `(context, claim)` to a **hallucination score in [0, 1]** (higher = more
likely hallucinated), so they are scored identically.

In [ ]:
from groundlens import verify

def groundlens_score(context, claim, question=None):
    """P(hallucination) from GroundLens evidence.

    We take the worst grounding signal over the answer's claims: 1 - min entailment
    across the NLI channel, falling back to semantic similarity; any explicit
    contradiction saturates the score. Deterministic, no threshold baked in.
    """
    rec = verify(claim, [("ctx", context)], question=question, bundle="base")
    nli = [e.score for e in rec.evidence if e.verifier_id == "groundlens.nli"]
    sem = [e.score for e in rec.evidence if e.verifier_id == "semantic.cosine"]
    contradicted = any(e.result == "contradicted" for e in rec.evidence)
    if nli:
        score = 1.0 - min(nli)
    elif sem:
        score = 1.0 - min(sem)
    else:
        score = 0.0
    if contradicted:
        score = max(score, 0.99)
    return float(score)

In [ ]:
import os, warnings, torch, transformers
from transformers import AutoModelForSequenceClassification

# Quiet the download progress bars and the cosmetic warnings (triton, model-type,
# 'a new version was downloaded'). These are noise, not errors.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
transformers.utils.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

assert transformers.__version__ < "5", (
    "transformers " + transformers.__version__ + " is 5.x; HHEM needs 4.x. "
    "Re-run the install cell, then RESTART THE RUNTIME and run from the top."
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pin the exact commit. HHEM runs custom code from the Hub (trust_remote_code); pinning
# the revision makes the download reproducible and stops it silently pulling new code.
# A hash-pinned load is also what a security review expects - the same guarantee
# GroundLens gives for its bundle.
HHEM_REVISION = "8e4a2e6e96c708cc76c2344f7e4757df2515292c"  # vectara/hallucination_evaluation_model

_hhem = AutoModelForSequenceClassification.from_pretrained(
    "vectara/hallucination_evaluation_model",
    trust_remote_code=True, revision=HHEM_REVISION)
_hhem = _hhem.to(DEVICE).eval()
print("transformers", transformers.__version__, "| HHEM device:", DEVICE,
      "| revision", HHEM_REVISION[:12])

def hhem_score(context, claim):
    """Vectara HHEM-2.1-Open: consistency prob in [0,1]; hallucination = 1 - consistency.
    Runs on GPU when the runtime has one; GroundLens stays on CPU by design."""
    with torch.no_grad():
        p = _hhem.predict([(context, claim)])  # premise, hypothesis
    if hasattr(p, "detach"):
        p = p.detach().cpu().flatten()[0].item()
    else:
        p = float(p[0]) if hasattr(p, "__len__") else float(p)
    return 1.0 - p


In [ ]:
# Optional: GPT-4o-as-judge - the stand-in for LangSmith / Phoenix / Braintrust / Ragas / DeepEval.
_judge_ready = RUN_LLM_JUDGE and bool(OPENAI_API_KEY)
if _judge_ready:
    from openai import OpenAI
    _client = OpenAI(api_key=OPENAI_API_KEY)

JUDGE_TOKENS = {"in": 0, "out": 0}
JUDGE_CALLS = 0

_JUDGE_PROMPT = (
    "You are a strict faithfulness checker. Given CONTEXT and a CLAIM, decide whether the "
    "CLAIM is fully supported by the CONTEXT. Reply with JSON only: "
    '{{"supported": true|false, "confidence": 0.0-1.0}}.\n\nCONTEXT:\n{ctx}\n\nCLAIM:\n{claim}'
)

def judge_score(context, claim):
    """P(hallucination) from an LLM judge: 1 - P(supported). Records token cost."""
    global JUDGE_CALLS
    if not _judge_ready:
        return None
    msg = _JUDGE_PROMPT.format(ctx=context[:6000], claim=claim[:2000])
    r = _client.chat.completions.create(
        model=OPENAI_MODEL, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": msg}])
    u = getattr(r, "usage", None)
    if u is not None:
        JUDGE_TOKENS["in"] += int(u.prompt_tokens); JUDGE_TOKENS["out"] += int(u.completion_tokens)
    JUDGE_CALLS += 1
    d = json.loads(r.choices[0].message.content)
    conf = float(d.get("confidence", 0.5))
    return (1.0 - conf) if d.get("supported") else conf

def judge_cost_per_1000():
    if not JUDGE_CALLS:
        return 0.0
    cost = JUDGE_TOKENS["in"]/1e6*PRICE_IN_PER_M + JUDGE_TOKENS["out"]/1e6*PRICE_OUT_PER_M
    return cost / JUDGE_CALLS * 1000


## 5 · Metrics

`FPR @ 95% recall` is the headline; AUROC and balanced accuracy for comparability. All computed the same way for every detector.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

R = int(TARGET_RECALL * 100)

def fpr_at_recall(y, scores, target_recall=TARGET_RECALL):
    y = np.asarray(y); scores = np.asarray(scores, dtype=float)
    fpr, tpr, _ = roc_curve(y, scores)
    ok = tpr >= target_recall
    return float(fpr[ok].min()) if ok.any() else 1.0

def best_bacc(y, scores):
    y = np.asarray(y); scores = np.asarray(scores, dtype=float)
    fpr, tpr, _ = roc_curve(y, scores)
    return float(((tpr + (1.0 - fpr)) / 2.0).max())

def op_threshold(y, scores, target=TARGET_RECALL):
    """Threshold that reaches >= target recall with the lowest false-positive rate."""
    y = np.asarray(y); s = np.asarray(scores, dtype=float)
    fpr, tpr, thr = roc_curve(y, s)
    ok = tpr >= target
    if not ok.any():
        return -np.inf   # flag everything
    idx = np.where(ok)[0]
    return float(thr[idx[np.argmin(fpr[idx])]])

def evaluate_full(y, scores, lat_ms=None):
    """Threshold-free metrics + everything at the fixed 95%-recall operating point."""
    y = np.asarray(y); s = np.asarray(scores, dtype=float)
    out = {"n": int(len(y)),
           "AUROC": round(float(roc_auc_score(y, s)), 3),
           "BACC": round(best_bacc(y, s), 3),
           f"FPR@{R}": round(fpr_at_recall(y, s), 3)}
    t = op_threshold(y, s)
    pred = (s >= t).astype(int)
    TP = int(((pred == 1) & (y == 1)).sum())
    FN = int(((pred == 0) & (y == 1)).sum())
    FP = int(((pred == 1) & (y == 0)).sum())
    P = int(y.sum())
    out[f"Prec@{R}"] = round(TP / (TP + FP), 3) if (TP + FP) else 0.0
    out[f"Rec@{R}"] = round(TP / P, 3) if P else 0.0
    out[f"FN@{R}"] = FN
    out["Review%"] = round(float(pred.mean()) * 100, 1)
    if lat_ms is not None and len(lat_ms):
        out["Lat_ms"] = round(float(np.median(lat_ms)), 1)
    return out


## 6 · Run the benchmark

Scores every example with every available detector, then computes the metrics per dataset. Per-example failures are skipped so one bad row never aborts the run.

In [ ]:
import time, gc
from tqdm.auto import tqdm

# Checkpoint on Google Drive, so a kernel restart (or an out-of-RAM crash) does not
# lose progress or re-pay for judge calls already made.
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/gl_bench"
except Exception:
    CKPT_DIR = "/content/gl_bench"   # fallback outside Colab
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

DETECTORS = {"GroundLens": groundlens_score,
             "HHEM-2.1-Open": lambda c, cl, q=None: hhem_score(c, cl)}
if _judge_ready:
    DETECTORS["GPT-4o-judge"] = lambda c, cl, q=None: judge_score(c, cl)

def _ckpt(name):
    return os.path.join(CKPT_DIR, name + "_n" + str(SAMPLE_SIZE) + ".jsonl")

def _complete(rec):
    return all(k in rec.get("scores", {}) for k in DETECTORS)

def run(rows, name):
    path = _ckpt(name)
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    d = json.loads(line); done[d["i"]] = d
        print(name, ": resuming from", len(done), "saved examples")
    fout = open(path, "a")
    for i, r in enumerate(tqdm(rows, desc=name)):
        if i in done and _complete(done[i]):
            continue
        ctx = (r["context"] or "")[:CONTEXT_CHARS]
        row_s, row_l, ok = {}, {}, True
        for k, fn in DETECTORS.items():
            t0 = time.perf_counter()
            try:
                v = fn(ctx, r["claim"], r.get("question"))
            except Exception:
                ok = False; break
            row_l[k] = (time.perf_counter() - t0) * 1000.0
            row_s[k] = v
        if not ok:
            continue
        rec = {"i": i, "y": r["y"], "ctype": r.get("ctype"),
               "scores": row_s, "latency": row_l}
        fout.write(json.dumps(rec) + "\n"); fout.flush()
        done[i] = rec
        if i % 25 == 0:
            gc.collect()
            if 'torch' in dir() and torch.cuda.is_available():
                torch.cuda.empty_cache()
    fout.close()
    order = sorted(k for k in done if _complete(done[k]))
    return {"y": [done[i]["y"] for i in order],
            "ctype": [done[i]["ctype"] for i in order],
            "scores": {k: [done[i]["scores"][k] for i in order] for k in DETECTORS},
            "latency": {k: [done[i]["latency"][k] for i in order] for k in DETECTORS}}

datasets_to_run = [("LLM-AggreFact", llm_aggrefact)] + ([("RAGTruth-QA", ragtruth)] if ragtruth else [])
results_raw = {}
for _name, _rows in datasets_to_run:
    results_raw[_name] = run(_rows, _name)
    gc.collect()


In [ ]:
import pandas as pd

cost1k = {d: 0.0 for d in DETECTORS}
if _judge_ready:
    cost1k["GPT-4o-judge"] = round(judge_cost_per_1000(), 4)

results = {}
rows = []
for ds, raw in results_raw.items():
    results[ds] = {}
    for det in DETECTORS:
        m = evaluate_full(raw["y"], raw["scores"][det], raw["latency"][det])
        m["Cost/1k$"] = cost1k.get(det, 0.0)
        results[ds][det] = m
        rows.append({"dataset": ds, "detector": det, **m})

headline = pd.DataFrame(rows)
headline


### Per sub-dataset (LLM-AggreFact)

How each detector does on each source set. Cells with too few examples or only one class are dropped; numbers on small `n` are noisy at low `SAMPLE_SIZE`.

In [ ]:
la = results_raw.get("LLM-AggreFact")
per_sub = None
if la:
    ct = np.array([c if isinstance(c, str) else "?" for c in la["ctype"]])
    y_all = np.array(la["y"])
    rows = []
    for sub in sorted(set(ct)):
        mask = ct == sub
        yy = list(y_all[mask])
        if mask.sum() < 20 or len(set(yy)) < 2:
            continue
        for det in DETECTORS:
            ss = list(np.array(la["scores"][det])[mask])
            m = evaluate_full(yy, ss)
            rows.append({"sub_dataset": sub, "detector": det, "n": m["n"],
                         "AUROC": m["AUROC"], f"FPR@{R}": m[f"FPR@{R}"],
                         f"Prec@{R}": m[f"Prec@{R}"], "Review%": m["Review%"]})
    per_sub = pd.DataFrame(rows)
per_sub


### Per hallucination type (RAGTruth)

At the threshold that gives 95% recall on the whole RAGTruth set, the **catch rate** for each labelled hallucination type (fraction of that type actually flagged). Only positives have a type, so this is recall by type, not FPR.

In [ ]:
rt = results_raw.get("RAGTruth-QA")
per_type = None
if rt:
    y = np.array(rt["y"])
    alltypes = sorted({t for c in rt["ctype"] if isinstance(c, list) for t in c})
    rows = []
    for det in DETECTORS:
        s = np.array(rt["scores"][det], dtype=float)
        thr = op_threshold(y, s)
        pred = (s >= thr).astype(int)
        for tp in alltypes:
            idx = [i for i, c in enumerate(rt["ctype"]) if isinstance(c, list) and tp in c]
            if not idx:
                continue
            rows.append({"hallucination_type": tp, "detector": det,
                         "n_pos": len(idx),
                         f"catch@{R}": round(int(pred[idx].sum()) / len(idx), 3)})
    per_type = pd.DataFrame(rows)
    if not per_type.empty:
        per_type = per_type.pivot(index="hallucination_type", columns="detector",
                                  values=f"catch@{R}")
per_type


### FPR @ 95% recall by detector

Lower is better: fewer false alarms at the recall a reviewer demands.

In [ ]:
import matplotlib.pyplot as plt

metric = f"FPR@{int(TARGET_RECALL*100)}"
fig, axes = plt.subplots(1, len(results), figsize=(5.2*len(results), 3.6), squeeze=False)
GREYS = {"GroundLens": "#111111", "HHEM-2.1-Open": "#9a9a9a", "GPT-4o-judge": "#c7c7c7"}
for ax, (ds_name, per_det) in zip(axes[0], results.items()):
    dets = list(per_det); vals = [per_det[d][metric] for d in dets]
    ax.bar(dets, vals, color=[GREYS.get(d, "#777") for d in dets], width=0.6)
    ax.set_title(ds_name, fontsize=11)
    ax.set_ylabel(metric); ax.set_ylim(0, 1)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.suptitle("False-positive rate at 95% recall (lower is better)", fontsize=12)
fig.tight_layout(); plt.show()

### Latency and cost

GroundLens and HHEM run locally, so cost is ~0 and latency is compute only. The LLM judge adds a per-call price and network round-trip - the slowest, priciest column, and the one that is not reproducible bit-for-bit.

In [ ]:
# Latency (median ms per claim) and cost (USD per 1000 verifications).
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.4, 3.6))
ds0 = next(iter(results))
dets = list(results[ds0])
lat = [results[ds0][d].get("Lat_ms", 0.0) for d in dets]
cost = [results[ds0][d].get("Cost/1k$", 0.0) for d in dets]
cols = [GREYS.get(d, "#777") for d in dets]
a1.bar(dets, lat, color=cols, width=0.6); a1.set_title(f"Median latency per claim ({ds0})")
a1.set_ylabel("ms"); a1.set_yscale("log")
a2.bar(dets, cost, color=cols, width=0.6); a2.set_title("Cost per 1000 verifications")
a2.set_ylabel("USD")
for ax, vals in ((a1, lat), (a2, cost)):
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:g}", ha="center", va="bottom", fontsize=9)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); plt.show()


## 7 · Reading the results

- **FPR @ 95% recall** is the operator's question: to catch 95% of hallucinations, how many
  correct answers does each detector wrongly flag? It is the metric a compliance team lives
  with, and — unlike balanced accuracy or the Vectara leaderboard — almost nobody reports it.
- **AUROC** is threshold-free and lets GroundLens's continuous grounding score compete
  cleanly against HHEM's.
- **Balanced accuracy** is the field's common currency (HHEM's card, the LLM-AggreFact
  leaderboard), included for direct comparability.

**What the head-to-head does not capture, and where GroundLens alone wins:** every row here
returns a number; only GroundLens returns a **signed, append-only evidence record** that
names the exact source span, the verifier and model hashes that ran, the policy that
decided, and the EU AI Act articles the decision maps to — verifiable offline by a third
party. HHEM, MiniCheck, LettuceDetect and an LLM judge give a score; none give an audit
trail. That is the layer a bank buys.

**The operational columns** (added at the fixed 95% recall point, which is where a bank actually runs a control): **Precision** and **Review%** say how much of the flagged workload is real versus wasted reviewer time; **FN@95** is the count of hallucinations still missed at that sensitivity; **Latency** and **Cost/1k** are the numbers a procurement team compares. The judge is the slowest and only paid column, and the only one without a reproducible record - the trade a compliance owner weighs against its flexibility.

**Honest caveats.** On free-text summarisation faithfulness (e.g. FaithBench) even SOTA
detectors sit near 55–58% balanced accuracy; a benchmark that only shows wins is not
trusted. GroundLens's exact numeric and rules verifiers dominate on quantitative claims;
on open-ended paraphrase its entailment and semantic channels carry the load, like everyone
else. The per-example score mapping (worst-claim entailment) is documented in the detector
cell so the comparison stays apples-to-apples.

**Cited, not run here** (see the analysis doc): MiniCheck / Bespoke-MiniCheck, LettuceDetect
(span-level peer), Patronus Lynx-8B/70B, Cleanlab TLM. Datasets to extend to: HaluBench
(domain breadth) and FaithBench (the hard set).

Full landscape, dataset table and sources: the *GroundLens benchmark analysis* document.
